In [10]:
import pandas as pd
from faker import Faker
from sqlalchemy import create_engine
from sqlalchemy.engine import Engine

In [11]:
def get_engine(user: str, password: str, host: str, port: int, schema: str) -> Engine:
    engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{schema}", pool_recycle=3600, pool_pre_ping=True)
    return engine


fake = Faker("ko_KR")
Faker.seed(42)


def get_result(query: str, engine: Engine):
    return pd.read_sql(query, engine)


def generate_customers(n: int) -> pd.DataFrame:
    data = []
    for _ in range(n):
        data.append({
            "name": fake.name(),
            "email": fake.unique.email(),
            "address": fake.address(),
            "phone_number": fake.phone_number(),
            "company": fake.company()
        })
    return pd.DataFrame(data)

In [12]:
primary_engine = get_engine("root", "root", "mysql-primary", 3306, "mmix")
replication_engine = get_engine("root", "root", "mysql-replication", 3307, "mmix")

In [13]:
primary_customers_dataframe = get_result("SELECT id, name, email, address, phone_number, company FROM customers", primary_engine)
primary_customers_dataframe.head(1)

,id,name,email,address,phone_number,company
0,1,김수민,seongmingim@example.net,울산광역시 중구 학동길 617,02-8386-3794,주식회사 하나전자


In [14]:
replication_customers_dataframe = get_result("SELECT id, name, email, address, phone_number, company FROM customers", replication_engine)
replication_customers_dataframe.head(1)

,id,name,email,address,phone_number,company


In [15]:
customers_dataframe = generate_customers(1000)
customers_dataframe.head()

,name,email,address,phone_number,company
0,김수민,seongmingim@example.net,울산광역시 중구 학동길 617,02-8386-3794,주식회사 하나전자
1,김진호,sanghyeon61@example.com,광주광역시 동구 반포대로 지하81 (정순강문마을),043-103-4131,(유) 우리첨단연구소
2,이정남,gangseongho@example.net,충청북도 횡성군 강남대835로 825-41,055-413-9537,유한회사 나루모두랩스
3,최상훈,zbae@example.com,충청북도 양구군 학동1길 611,032-669-7848,(주) 푸른항공
4,황순자,jinu70@example.net,대구광역시 구로구 가락거리 204 (준서김마을),02-9570-1543,주식회사 제나기획


In [16]:
customers_dataframe.to_sql(name="customers", con=primary_engine, if_exists="append", index=False, chunksize=1000, method="multi")

1000

In [17]:
replication_customers_dataframe = get_result("SELECT id, name, email, address, phone_number, company FROM customers", replication_engine)
replication_customers_dataframe.head(1)

,id,name,email,address,phone_number,company
